# Iteration 8: today's rule fails on fresh NUDT photos; a DINO-z veto; blind confirmation

**What happened.** On 200 NUDT photos that no earlier test had used (fresh 404: drawn from outside the 500-photo pool every earlier seed came
from), today's best rule (bar 8 or agreement, then the 27-query escalation at bar 11) made **4 wrong answers** (53.43). All four are on the city
flights (look-alike buildings): two answered at the bar with 8 and 9 inliers, two by agreement. So the earlier "0 wrong on every set" did not
hold on unseen photos, and every later rule must be confirmed on photos that were not used to choose it.

**Hypothesis.** A false place among look-alike buildings can collect a handful of geometric inliers, but its *content* differs from the photo,
so DINO scores it low compared with the other candidates of the shortlist. The 4 wrong answers all have DINO z 0.8-1.3, while right answers
mostly sit far higher (Iteration 2: z separates right from wrong with AUC 0.98 official / 0.90 NUDT). **Rule:** an answer (at any step) also
needs DINO z >= z0 or a clearly safe inlier count (>= 15).

**Protocol.** The veto and the alternatives were chosen on the four seen sets (official, NUDT 101, 202, 404). The settings were then frozen
(`results/frozen_rules.json`, time-stamped) **before** a second fresh NUDT sample (505: 300 photos outside the pool and outside 404) was loaded
or scored. Section 5 scores the frozen rules on 505 once.

In [ ]:
import os, sys, json
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 40)
from train.checks.replay import load_set, truncate_m, score, paired_bootstrap, headroom, query_ids, WIDE
from train.checks.query_policy import run_policy
from train.datasets import nudt_data
OUT = ROOT / "tests/Test_Iteration_6/results"; DUMPS = OUT / "rich_dump"
SEEN = {"official": "official", "nudt101": "nudt_seed101", "nudt202": "nudt_seed202", "nudt404": "nudt_fresh404"}
sets = {n: load_set(DUMPS / f) for n, f in SEEN.items()}
frozen = json.load(open(OUT / "frozen_rules.json"))
print("frozen at", frozen["frozen_at"]); print({k: len(v) for k, v in sets.items()})

## 1. The four wrong answers on fresh NUDT 404

In [ ]:
items = nudt_data.read_split(nudt_data.configure(), "test")
from train.checks.replay import decide_current
rows = []
for pid, ph in sets["nudt404"].items():
    r = decide_current(ph)
    if r["points"] < 0:
        h = headroom(ph, query_ids(ph, WIDE))
        rows.append(dict(photo=pid, flight=items[pid]["flight"], height=round(ph["height"]), stage=r["stage"], inliers=r["inliers"], agree=r["agree"],
                         err_sigma=round(r["err_sigma"], 1), right_place_in_27=h["right_present"], right_best_inliers=h["right_best"]))
display(pd.DataFrame(rows))
rows_seen = pd.concat([pd.read_csv(OUT / ("it2_rows_%s.csv" % n)) for n in SEEN], ignore_index=True)
ans = rows_seen[(rows_seen.step == "std6") & rows_seen.has_choice & ((rows_seen.inliers >= 8) | (rows_seen.agree >= 2))]
ans = ans.assign(wrong=ans.err_sigma > 8)
print("DINO z of today's 6-query answers: right answers by set, and the wrong ones")
display(ans.groupby(["data", "wrong"]).z.describe()[["count", "min", "25%", "50%"]].round(2))
print("right answers with z < 1.3 and < 15 inliers (what the veto would cost):",
      ans[~ans.wrong & (ans.z < 1.3) & (ans.inliers < 15)].groupby("data").size().to_dict())

## 2. The alternatives on the four seen sets

Grids saved by the exploratory runs of this iteration (`it8_rule_grid_seen_sets.csv`: stricter bar / agreement; `it8_veto_grid_seen_sets.csv`:
the z and hull veto). Below: today's rule, the best 0-wrong strict rule, the best 0-wrong veto.

In [ ]:
grid = pd.read_csv(OUT / "it8_rule_grid_seen_sets.csv"); veto = pd.read_csv(OUT / "it8_veto_grid_seen_sets.csv")
grid["w"] = grid[[c for c in grid if c.endswith("_w")]].sum(1)
cols = ["official", "nudt101", "nudt202", "nudt404"]
print("stricter bar / agreement, 0 wrong on all four, best first:")
display(grid[grid.w == 0].sort_values("official", ascending=False).head(5)[["bar", "q", "amin", "esc"] + cols])
print("z / hull veto on top of today's rule, 0 wrong on all four, best first:")
display(veto[veto.w == 0].sort_values("mean", ascending=False).head(6)[["z0", "hull0", "safe"] + cols])
print("how flat is z0 (hull off, safe 15)?")
display(veto[(veto.hull0 < 0) & (veto.safe == 15)][["z0"] + cols + ["w"]])

## 3. Learned verifier as a veto (leave-one-set-out)

Exploratory run of this iteration: the logistic / GBM verifier as a veto on today's answers plus late answers, trained on three sets, cuts chosen
there (0 wrong), tested on the fourth. Result (logistic, all features): official +1.58 [+0.32, +3.01] 0 wrong; NUDT 404 4 -> 0 wrong
+2.97 [+0.60, +5.94]; but NUDT 101 **1 wrong** when it is held out; GBM and count-free variants also leave 1-2 wrong on some held-out set.
With 4 wrong answers in ~1100 photos there is too little to learn a safe boundary: the learned veto is not adopted (needs more data). The
single-feature z veto has one threshold, a stated reason, and a flat region (z0 1.3-1.5).

## 4. Frozen policies on the seen sets (PC: full rule; Jetson: capped)

`jetson_today` replays the submitted code (M = 5, bar 8, no agreement, second pass, cap 10). `J_M10` / `J_M5` are the Jetson-feasible
proposals. Jetson seconds = mean queries x seconds per query (2.2 s at M = 5; M = 10 assumed 2.6 s).

In [ ]:
g = pd.read_csv(OUT / "it1_greedy_order.csv"); ORDER = [(float(h), float(s)) for h, s in zip(g.heading, g.scale)]
SP = [(h, s) for s in (1.0, 1.05) for h in (-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0)]
POLICIES = {
    "jetson_today": (5, 2.2, dict(agree_min=99, escalation_bar=8, wide_order=SP, max_queries=10)),
    "today (PC, no cap)": (10, None, dict()),
    "today + z-veto (PC, no cap)": (10, None, dict(z_veto=tuple(frozen["rules"]["today_zveto"]["z_veto"]))),
    "strict bar 10, 3 agree (PC)": (10, None, dict(bar=10, agree_min=3)),
    "J_M10 (cap 8, greedy, z-veto)": (10, 2.6, dict(wide_order=ORDER, max_queries=8, z_veto=(1.3, 15))),
    "J_M5 (cap 9, greedy, agree>=4, z-veto)": (5, 2.2, dict(wide_order=ORDER, max_queries=9, agree_query_inliers=4, z_veto=(1.3, 15))),
}
def run_all(photos, M, kw):
    return pd.DataFrame([dict(photo=pid, **run_policy(truncate_m(p, M) if M < 10 else p, **kw)) for pid, p in photos.items()])
def table(sets_):
    tabs = {}; rows = []
    for name, (M, s_q, kw) in POLICIES.items():
        for n, ph in sets_.items():
            t = run_all(ph, M, kw); tabs[(name, n)] = t
            rows.append(dict(policy=name, set=n, score=round(score(t.points), 2), wrong=int((t.points < 0).sum()), mean_q=round(t.n_queries.mean(), 2),
                             jetson_s=round(t.n_queries.mean() * s_q, 1) if s_q else np.nan))
    res = pd.DataFrame(rows)
    for name in POLICIES:
        for n in sets_:
            gain = paired_bootstrap(tabs[("jetson_today", n)].points.to_numpy(), tabs[(name, n)].points.to_numpy())
            gain2 = paired_bootstrap(tabs[("today (PC, no cap)", n)].points.to_numpy(), tabs[(name, n)].points.to_numpy())
            res.loc[(res.policy == name) & (res.set == n), "vs_jetson_today"] = "%+.2f [%+.2f, %+.2f]" % gain
            res.loc[(res.policy == name) & (res.set == n), "vs_today"] = "%+.2f [%+.2f, %+.2f]" % gain2
    return res, tabs
seen_res, seen_tabs = table(sets)
display(seen_res.pivot(index="policy", columns="set", values=["score", "wrong", "mean_q"]))
display(seen_res.pivot(index="policy", columns="set", values="vs_jetson_today"))
seen_res.to_csv(OUT / "it8_policies_seen_sets.csv", index=False)

## 5. Blind confirmation on fresh NUDT 505 (300 photos, loaded here for the first time)

Nothing above used these photos. The frozen settings are scored once; no setting is changed after this cell.

In [ ]:
blind_folder = DUMPS / "nudt_fresh505"
blind = {"nudt505": load_set(blind_folder)}
print("blind photos:", len(blind["nudt505"]), "| file written", pd.Timestamp(os.path.getmtime(blind_folder), unit="s"))
blind_res, blind_tabs = table(blind)
display(blind_res[["policy", "score", "wrong", "mean_q", "jetson_s", "vs_jetson_today", "vs_today"]])
blind_res.to_csv(OUT / "it8_policies_blind_nudt505.csv", index=False)

## 6. Five sets side by side, step by step (written after the blind check)

Each piece of the decision added in turn, and the Jetson replay with the free pieces added. **G** keeps the submitted second-pass order (the
frozen J_M5 used the greedy order of Iteration 1, which lost on the blind set): this swap was made after section 5, and it removes a tuned piece.

In [ ]:
all_sets = dict(sets, nudt505=blind["nudt505"])
STEPS = [("A bar 8 only, 6 queries (M10)", 10, dict(agree_min=99, max_queries=6)),
         ("B + agreement (6 queries)", 10, dict(max_queries=6)),
         ("C + escalation 27 @11 = today", 10, dict()),
         ("D today + z-veto", 10, dict(z_veto=(1.3, 15))),
         ("E Jetson today (M5, 2nd pass @8, cap 10)", 5, dict(agree_min=99, escalation_bar=8, wide_order=SP, max_queries=10)),
         ("F E + agreement q>=4", 5, dict(agree_query_inliers=4, escalation_bar=8, wide_order=SP, max_queries=10)),
         ("G E + agreement q>=4 + z-veto", 5, dict(agree_query_inliers=4, escalation_bar=8, wide_order=SP, max_queries=10, z_veto=(1.3, 15)))]
tabs6, rows = {}, []
for name, M, kw in STEPS:
    row = dict(policy=name)
    for n, ph in all_sets.items():
        t = run_all(ph, M, kw); tabs6[(name, n)] = t
        row[n] = "%.2f / %dw / %.1fq" % (score(t.points), (t.points < 0).sum(), t.n_queries.mean())
    rows.append(row)
final = pd.DataFrame(rows); display(final); final.to_csv(OUT / "final_summary_five_sets.csv", index=False)
pairs = [(STEPS[0][0], STEPS[1][0]), (STEPS[1][0], STEPS[2][0]), (STEPS[2][0], STEPS[3][0]), (STEPS[4][0], STEPS[5][0]), (STEPS[4][0], STEPS[6][0])]
display(pd.DataFrame([dict(change="%s -> %s" % (a[:1], b[:1]), **{n: "%+.2f [%+.2f, %+.2f]" % paired_bootstrap(tabs6[(a, n)].points, tabs6[(b, n)].points) for n in all_sets})
                      for a, b in pairs]))
print("wrong photos, E vs G:", {n: ([i for i, p in zip(tabs6[(STEPS[4][0], n)].photo, tabs6[(STEPS[4][0], n)].points) if p < 0],
                                    [i for i, p in zip(tabs6[(STEPS[6][0], n)].photo, tabs6[(STEPS[6][0], n)].points) if p < 0]) for n in all_sets})

## 7. The attitude correction on top (official camera, held out by flight)

In [ ]:
from sklearn.model_selection import GroupKFold
from train.checks.pose_bias import fit as fit_bias, apply as apply_bias
from train.checks.replay import points as pts
def with_positions(name):
    M, kw = [(m, k) for nm, m, k in STEPS if nm == name][0]
    rows = []
    for pid, p in sets["official"].items():
        r = run_policy(truncate_m(p, M) if M < 10 else p, **kw)
        rows.append(dict(photo=pid, answered=r["answered"], points=r["points"], err_sigma=r["err_sigma"],
                         ex=(r["x"] - p["truth"][0]) if r["x"] is not None else 0.0, ey=(r["y"] - p["truth"][1]) if r["y"] is not None else 0.0,
                         sigma=p["sigma"], h=p["height"], yaw=p["yaw"], pitch=p["pitch"], roll=p["roll"]))
    t = pd.DataFrame(rows); t["flight"] = t.h.round(0); return t
def corrected(t):
    newp = t.points.to_numpy().copy()
    for tr, te in GroupKFold(5).split(t, groups=t.flight):
        a = t.iloc[tr]; m = fit_bias(a[a.answered], ("angle", "const")); b = t.iloc[te]; ex, ey = apply_bias(b, m)
        e = np.hypot(ex, ey) / b.sigma.to_numpy(); newp[te] = [pts(x) if ans else 0 for x, ans in zip(e, b.answered)]
    return newp
E_, G_ = with_positions(STEPS[4][0]), with_positions(STEPS[6][0])
for nm, t in (("E Jetson today", E_), ("G", G_)):
    cp = corrected(t)
    print("%s: %.2f -> with attitude %.2f, gain %+.2f [%+.2f, %+.2f], wrong %d" % ((nm, score(t.points), score(cp)) + paired_bootstrap(t.points, cp) + (int((cp < 0).sum()),)))
print("G + attitude vs Jetson today: %+.2f [%+.2f, %+.2f]" % paired_bootstrap(E_.points, corrected(G_)))
m = fit_bias(G_[G_.answered], ("angle", "const"))
print("coefficients fitted on all official G answers: forward %.3f deg, lateral %.3f deg, constant (%.2f, %.2f) m"
      % (np.degrees(np.arctan(m["coef"][0])), np.degrees(np.arctan(m["coef"][1])), m["coef"][2], m["coef"][3]))